# MMIT-DDPM — Training Notebook
**Multilateral Medical Image Translation with Class and Structure Supervised Diffusion-Based Model**

Paper: [Computers in Biology and Medicine, 2024](https://doi.org/10.1016/j.compbiomed.2024.109501)

This notebook runs the full MMIT-DDPM pipeline on BraTS data:
1. Clone repo & install dependencies
2. Download BraTS dataset (Synapse)
3. Train the diffusion model
4. Sample translated images

In [ ]:
import os

REPO_URL  = 'https://github.com/hashirama21/mmit-ddpm.git'
REPO_NAME = 'mmit-ddpm'

if not os.path.isdir(REPO_NAME):
    os.system(f'git clone {REPO_URL}')

if os.path.basename(os.getcwd()) != REPO_NAME:
    os.chdir(REPO_NAME)

print('Working directory:', os.getcwd())
os.system('pip install -q -r requirements.txt')
print('Dependencies installed.')

In [ ]:
# Download BraTS dataset from Synapse (https://www.synapse.org)
# Set SKIP_DOWNLOAD = True if ./data/ is already populated.

SKIP_DOWNLOAD = False

if not SKIP_DOWNLOAD:
    import getpass
    os.system('pip install -q synapseclient')
    import synapseclient

    syn_token = os.environ.get('SYNAPSE_AUTH_TOKEN') or getpass.getpass('Synapse auth token: ')
    syn = synapseclient.Synapse()
    syn.login(authToken=syn_token)

    DATASETS = {
        'training': 'syn51514132',
        'testing':  'syn51514135',
    }

    os.makedirs('./data', exist_ok=True)
    for split, syn_id in DATASETS.items():
        out_dir = f'./data/{split}'
        if os.path.isdir(out_dir) and os.listdir(out_dir):
            print(f'[{split}] already present — skipped')
            continue
        print(f'[{split}] downloading {syn_id} ...')
        syn.get(syn_id, downloadLocation=out_dir, ifcollision='overwrite.local')
        print(f'[{split}] done')
else:
    print('SKIP_DOWNLOAD=True — expecting data in ./data/')

In [ ]:
# Training flags — edit as needed
MODEL_FLAGS     = '--image_size 256 --num_channels 128 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16'
DIFFUSION_FLAGS = '--diffusion_steps 1000 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False'
TRAIN_FLAGS     = '--lr 1e-4 --batch_size 10'

cmd = f'python scripts/train_translation.py --data_dir ./data/training {TRAIN_FLAGS} {MODEL_FLAGS} {DIFFUSION_FLAGS}'
print('Running:', cmd)
os.system(cmd)

In [ ]:
# Sampling — generates 5 translated outputs per test slice
MODEL_PATH  = './results/savedmodel.pt'   # path to the saved checkpoint
NUM_ENSEMBLE = 5

cmd = (
    f'python scripts/sampling_translation.py'
    f' --data_dir ./data/testing'
    f' --model_path {MODEL_PATH}'
    f' --num_ensemble {NUM_ENSEMBLE}'
    f' {MODEL_FLAGS} {DIFFUSION_FLAGS}'
)
print('Running:', cmd)
os.system(cmd)